# 📰 MINI PROJECT: TEXT SUMMARIZATION BAHASA INDONESIA
## Komparasi Extractive Summarization: TF-IDF Sentence Ranking vs. TextRank Algorithm
**Tugas Lanjutan Pertemuan 11 & 12 — Pemrosesan Bahasa Alami (NLP)**  
**Program Studi:** S1 Teknik Informatika — Universitas Negeri Malang (UM)  
**Dosen Pengampu:** Prof. Ir. Aji Prasetya Wibawa, S.T., M.M.T., Ph.D. & M. Zainal Arifin, S.Si., M.Kom., Ph.D.  
**Mentor Asisten:** Agung Bella Putra Utama & Fadia Irsania Putri  

---

### 👥 Pembagian Peran & Job Desk Tim (5 Anggota)

| No | NIM | Nama Mahasiswa | Peran & Tanggung Jawab Utama |
| :-: | :---: | :--- | :--- |
| **1** | `240535600363` | **ASYAM SAKHI MAULANA** | **Pengembang Model 1 (TF-IDF Sentence Ranking):**<br>• Preprocessing linguistik: segmentasi kalimat regex & normalisasi kata.<br>• Pemodelan matriks TF-IDF kalimat & pembobotan salience score berbobot.<br>• Seleksi Top-K kalimat penting & rekonstruksi ringkasan kronologis.<br>• Pengujian & visualisasi inferensi Model 1 *(Status: 100% Selesai & Valid)*. |
| **2** | `240535607403` | **AGUSTINUS MARIA AGUNG HERIYANTO** | **Pengembang Model 2 (TextRank Algorithm):**<br>• Preprocessing graf & segmentasi kalimat untuk node graf.<br>• Konstruksi matriks kemiripan kosinus & graf NetworkX.<br>• Perhitungan sentralitas PageRank & rekonstruksi ringkasan TextRank.<br>• *(Status: Starter Template & Instruksi TODO)*. |
| **3** | `240535604624` | **CHRISTIAN LIONEL DAUN DARMA** | **Cari Dataset & Penulisan Kesimpulan:**<br>• Mencari & memuat dataset IndoSum resmi (Kata.ai) langsung di Colab.<br>• Menulis analisis evaluasi komparatif & kesimpulan akhir (Slide 20 PDF). |
| **4** | `240535601034` | **ALBI ADWITIYA CHANDRA** | **Mini Project Text Generation:** Pengembang Model 1 (LSTM Language Model). |
| **5** | `240535610197` | **INDAH HARIYANTI** | **Mini Project Text Generation:** Pengembang Model 2 (GRU Language Model). |

---

### 🎯 Tujuan & Aspek Komparasi Sesuai Slide 20 PDF
Masing-masing kelompok wajib membandingkan kedua model ekstraktif dari sisi:
1. **Skor ROUGE-1, ROUGE-2, dan ROUGE-L** terhadap ringkasan acuan manusia (*gold summary*)
2. **Kualitas ringkasan** dan kelancaran alur baca (*coherence*)
3. **Kemampuan mempertahankan informasi penting** (*salience preservation*)
4. **Konsistensi pemilihan kalimat** dan eliminasi redundansi
5. **Waktu proses komputasi** (*processing latency*)
6. **Contoh ringkasan terbaik dan terburuk**
7. **Analisis error & kelemahan masing-masing metode**
8. **Diskusi Pertanyaan Utama:** *Apakah model dengan skor ROUGE tertinggi selalu menghasilkan ringkasan yang paling baik secara makna?*
9. **Bonus:** Komparasi hasil extractive summarization dengan model abstractive summarization.


## 1. Setup Lingkungan & Import Library
Instalasi pustaka `rouge-score`, `datasets`, `networkx`, `Sastrawi`, dan `gdown` untuk pemrosesan NLP, graf, dan pengunduhan korpus resmi langsung di Colab.


In [ ]:
# Instalasi library pendukung jika belum tersedia di runtime Colab
!pip install --quiet rouge-score datasets networkx Sastrawi gdown

import os
import re
import time
import glob
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("✅ Seluruh pustaka pendukung berhasil diimpor!")


---
## 2. Dataset IndoSum: Pencarian & Pemuatan Korpus Berita Resmi
**Penanggung Jawab:** **CHRISTIAN LIONEL DAUN DARMA** (NIM: `240535604624`) — *Tugas: Cari & Load Dataset*  
**Sumber Korpus Resmi:** [https://github.com/kata-ai/indosum](https://github.com/kata-ai/indosum) *(Kata.ai Benchmark Dataset)*

Dataset **IndoSum** diunduh langsung dari repositori resmi secara murni di Google Colab tanpa data *dummy* tiruan. Berkas resmi dalam format JSON Lines (`.jsonl`) dibaca dan dikonversi langsung ke struktur tabel analisis.


In [ ]:
# ==============================================================================
# BAGIAN 2: PENCARIAN & PEMUATAN DATASET INDOSUM RESMI (KATA.AI)
# Penanggung Jawab: Christian Lionel Daun Darma (240535604624)
# ==============================================================================

# 1. Unduh dataset resmi indosum.zip langsung dari Google Drive ID resmi Kata.ai jika belum ada
if not os.path.exists("indosum.zip") and not glob.glob("**/test.01.jsonl", recursive=True):
    print("⏳ [Daun] Mengunduh korpus benchmark resmi IndoSum dari repositori Kata.ai...")
    !gdown --id 1OgYbPfXFAv3TbwP1Qcwt_CC9cVWSJaco -O indosum.zip

# 2. Ekstrak arsip resmi indosum.zip
if os.path.exists("indosum.zip") and not glob.glob("**/test.01.jsonl", recursive=True):
    print("📦 [Daun] Mengekstrak arsip resmi indosum.zip...")
    with zipfile.ZipFile("indosum.zip", "r") as zip_ref:
        zip_ref.extractall("indosum_data")

# 3. Cari dan baca file test resmi (test.01.jsonl)
jsonl_candidates = glob.glob("**/test.01.jsonl", recursive=True) or glob.glob("**/*.jsonl", recursive=True)

records = []
if jsonl_candidates:
    target_file = jsonl_candidates[0]
    print(f"📖 [Daun] Membaca korpus berita asli dari: {target_file}")
    with open(target_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 30:  # Batasi 30 artikel berita resmi untuk benchmarking efisien
                break
            item = json.loads(line.strip())
            # Rekonstruksi kalimat artikel dari struktur hierarki IndoSum: paragraphs -> sentences -> words
            raw_article = ' '.join(' '.join(token for token in sent) for p in item.get('paragraphs', []) for sent in p)
            raw_summary = ' '.join(' '.join(token for token in sent) for sent in item.get('summary', []))
            
            records.append({
                "id": f"indosum_{item.get('id', i+1)}",
                "category": item.get("category", "berita"),
                "title": f"Artikel IndoSum #{item.get('id', i+1)} ({item.get('category', 'Umum')})",
                "article": raw_article.strip(),
                "gold_summary": raw_summary.strip()
            })
    df_indosum = pd.DataFrame(records)
else:
    # Alternatif cadangan: unduh langsung lewat library datasets Hugging Face
    print("📥 [Daun] Mengunduh langsung lewat mirror SEACrowd IndoSum di Hugging Face...")
    from datasets import load_dataset
    hf_ds = load_dataset("SEACrowd/indosum", "indosum_id_seacrowd", split="test[:30]")
    df_indosum = pd.DataFrame(hf_ds)

# 4. Hitung statistik karakteristik korpus IndoSum (Eksplorasi Data oleh Daun)
df_indosum["article_word_count"] = df_indosum["article"].apply(lambda x: len(x.split()))
df_indosum["gold_word_count"] = df_indosum["gold_summary"].apply(lambda x: len(x.split()))
df_indosum["compression_ratio"] = (df_indosum["gold_word_count"] / df_indosum["article_word_count"]).round(4)

print(f"\n🎉 SUKSES! Total {len(df_indosum)} artikel berita resmi IndoSum berhasil dimuat secara murni!")
print(f" • Rata-rata panjang kata artikel berita : {df_indosum['article_word_count'].mean():.1f} kata")
print(f" • Rata-rata panjang kata ringkasan acuan: {df_indosum['gold_word_count'].mean():.1f} kata")
print(f" • Rata-rata Rasio Kompresi (Gold/Asli)  : {df_indosum['compression_ratio'].mean() * 100:.1f} %")

display(df_indosum[["id", "category", "article_word_count", "gold_word_count", "compression_ratio"]].head(10))


In [ ]:
# Inspeksi pratinjau teks artikel berita resmi pertama hasil unduhan Daun
print("="*80)
print(f"PRATINJAU ARTIKEL BERITA RESMI: {df_indosum.iloc[0]['title']}")
print("="*80)
print("[Artikel Asli]:")
print(df_indosum.iloc[0]['article'][:500] + "...")
print("\n[Ringkasan Acuan Manusia / Gold Summary]:")
print(df_indosum.iloc[0]['gold_summary'])


---
## 3. Model 1: Extractive Summarization — TF-IDF Sentence Ranking
**Penanggung Jawab:** **ASYAM SAKHI MAULANA** (NIM: `240535600363`)  
**Peran & Fokus:** Preprocessing Linguistik Internal (Segmentasi Kalimat & Case Folding), Pemodelan Matriks TF-IDF Kalimat, Perhitungan Salience Score Berbobot, Seleksi Top-K Kalimat Penting, & Rekonstruksi Ringkasan Kronologis.

---

### 🧠 Landasan Teori & Pipeline TF-IDF Sentence Ranking

Model 1 memandang peringkasan teks sebagai masalah pemilihan subset kalimat ($S^* \subset S$) yang memuat konsentrasi informasi leksikal tertinggi dari artikel berita asli.

```text
Pipeline Lengkap Model 1 (TF-IDF Sentence Ranking):
[Teks Berita Asli]
       │
       ▼ Preprocessing: Segmentasi Kalimat (Regex Sentence Tokenizer)
[Daftar Kalimat Dokumen: s_1, s_2, ..., s_N]
       │
       ▼ Normalisasi Teks (Case Folding, Punctuation Filter, Stopword Removal ID)
[Kalimat Bersih Ter-Normalisasi]
       │
       ▼ Pembentukan Matriks TF-IDF (Tiap kalimat dianggap sebagai dokumen mini)
[Matriks TF-IDF: (N_kalimat × M_kata)]
       │
       ▼ Perhitungan Salience Score Kalimat: Score(s_i) = ∑ TF-IDF(w, s_i) / √(panjang(s_i))
[Vektor Skor Kepentingan Tiap Kalimat]
       │
       ▼ Pemilihan Top-K Kalimat Terbaik (K = 2 atau 3 kalimat)
[Indeks Kalimat Terpilih]
       │
       ▼ Chronological Re-ordering (Mengurutkan kembali sesuai urutan asli di teks)
[Ringkasan Ekstraktif Akhir yang Koheren]
```

#### 1. Rumus Pembobotan Kata (TF-IDF)
Setiap kata $w$ dalam kalimat $s_i$ diberi bobot berdasarkan frekuensi kemunculannya dalam kalimat tersebut (*Term Frequency*) dan kelangkaannya di seluruh dokumen (*Inverse Document Frequency*):
$$\text{TF-IDF}(w, s_i, D) = \text{TF}(w, s_i) \times \left( \log\left( \frac{N + 1}{\text{DF}(w) + 1} \right) + 1 \right)$$

#### 2. Perhitungan Skor Kalimat & Normalisasi Panjang (*Length Normalization*)
Jika kita hanya menjumlahkan nilai TF-IDF tanpa normalisasi, kalimat yang sangat panjang akan selalu memperoleh skor paling tinggi secara bias. Oleh karena itu, diterapkan normalisasi akar panjang kata:
$$\text{Score}(s_i) = \frac{\sum_{w \in s_i} \text{TF-IDF}(w, s_i)}{\sqrt{|s_i|}}$$

#### 3. Rekonstruksi Kronologis (*Chronological Re-ordering*)
Setelah memilih $K$ kalimat dengan skor tertinggi, urutan kalimat dikembalikan mengikuti posisi awal kemunculannya pada artikel berita aslinya. Hal ini menjamin transisi logika antar-kalimat tetap wajar dan tidak terbalik-balik.


In [ ]:
# ==============================================================================
# BAGIAN 3.1: IMPLEMENTASI KELAS TF-IDF SENTENCE RANKING SUMMARIZER
# Pengembang: Asyam Sakhi Maulana (240535600363)
# ==============================================================================

# Daftar stopword Bahasa Indonesia baku yang difilter agar tidak mendominasi bobot TF-IDF
STOPWORDS_ID = {
    'yang', 'dan', 'di', 'dari', 'ini', 'itu', 'untuk', 'pada', 'ke', 'dengan', 'adalah',
    'karena', 'atas', 'sebagai', 'oleh', 'juga', 'dalam', 'akan', 'telah', 'saat', 'dapat',
    'atau', 'serta', 'namun', 'ia', 'dia', 'mereka', 'kami', 'kita', 'tersebut', 'bisa',
    'ada', 'tidak', 'lebih', 'sudah', 'hingga', 'bagi', 'antara', 'menjadi', 'tahun', 'hari'
}

class TFIDFSummarizer:
    """
    Model Extractive Summarization berbasis TF-IDF Sentence Ranking.
    Dikembangkan oleh: Asyam Sakhi Maulana (240535600363)
    """
    def __init__(self, stopwords=STOPWORDS_ID, length_penalty_power=0.5):
        self.stopwords = list(stopwords)
        self.power = length_penalty_power
        
    def split_into_sentences(self, text):
        """
        [Modul Preprocessing Asyam]
        Memecah teks paragraf berita menjadi daftar kalimat berdasarkan tanda baca akhir.
        """
        raw_sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s.strip() for s in raw_sentences if len(s.strip().split()) >= 3]
        
    def clean_sentence(self, sentence):
        """
        [Modul Preprocessing Asyam]
        Membersihkan karakter non-alfanumerik dan case folding untuk perhitungan TF-IDF.
        """
        s = str(sentence).lower()
        s = re.sub(r'[^a-z0-9\s]', ' ', s)
        return re.sub(r'\s+', ' ', s).strip()
        
    def summarize(self, text, num_sentences=2):
        """
        Mengekstrak k-kalimat paling penting dari artikel berita menggunakan skor TF-IDF.
        
        Parameters:
            text (str): Teks artikel berita lengkap.
            num_sentences (int): Jumlah kalimat ringkasan yang diinginkan (default: 2).
            
        Returns:
            summary_text (str): Teks ringkasan ekstraktif berurutan.
            ranking_details (list of dict): Rincian kalimat, skor, dan peringkatnya.
        """
        # 1. Preprocessing: Segmentasi kalimat internal
        sentences = self.split_into_sentences(text)
        if len(sentences) <= num_sentences:
            return text.strip(), [{"original_pos": i, "sentence": s, "score": 1.0, "is_selected": True} for i, s in enumerate(sentences)]
            
        # 2. Normalisasi kalimat untuk matriks TF-IDF
        cleaned_sentences = [self.clean_sentence(s) for s in sentences]
        
        # 3. Fitting TfidfVectorizer pada kalimat-kalimat artikel
        vectorizer = TfidfVectorizer(
            stop_words=self.stopwords,
            ngram_range=(1, 1),
            sublinear_tf=True
        )
        try:
            tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)
        except ValueError:
            # Fallback jika kalimat terlalu pendek/hanya stopword
            return ' '.join(sentences[:num_sentences]), []
            
        # 4. Hitung skor salience: jumlah bobot TF-IDF dibagi akar panjang kata kalimat
        sentence_lengths = np.array([max(len(s.split()), 1) for s in cleaned_sentences])
        raw_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()
        salience_scores = raw_scores / (sentence_lengths ** self.power)
        
        # 5. Peringkat kalimat dan ambil Top-K kalimat dengan skor tertinggi
        ranked_indices = np.argsort(salience_scores)[::-1]
        top_k_indices = ranked_indices[:num_sentences]
        
        # 6. Rekonstruksi kronologis: urutkan kembali sesuai urutan kemunculan di teks asli
        chronological_indices = sorted(top_k_indices)
        selected_sentences = [sentences[i] for i in chronological_indices]
        
        summary_text = ' '.join(selected_sentences)
        
        ranking_details = [
            {
                "original_pos": i,
                "sentence": sentences[i],
                "score": round(float(salience_scores[i]), 4),
                "is_selected": (i in top_k_indices)
            }
            for i in range(len(sentences))
        ]
        
        return summary_text, ranking_details

# Inisialisasi model TF-IDF Summarizer Asyam
tfidf_model = TFIDFSummarizer()
print("✅ Model 1: TFIDFSummarizer (Asyam Sakhi Maulana) siap digunakan!")


In [ ]:
# ==============================================================================
# BAGIAN 3.2: PENGUJIAN & INSPEKSI INFERENSI MODEL 1 PADA SAMPEL ARTIKEL
# Pengembang: Asyam Sakhi Maulana (240535600363)
# ==============================================================================

# Uji Model 1 pada artikel pertama (Berita Jalan Tol Ekonomi)
contoh_artikel = df_indosum.iloc[0]["article"]
contoh_gold = df_indosum.iloc[0]["gold_summary"]
contoh_judul = df_indosum.iloc[0]["title"]

t_start = time.time()
ringkasan_m1, rincian_skor = tfidf_model.summarize(contoh_artikel, num_sentences=2)
waktu_m1 = time.time() - t_start

print("="*80)
print("HASIL UJI COBA MODEL 1 (TF-IDF SENTENCE RANKING) - ASYAM SAKHI MAULANA")
print("="*80)
print(f"Judul Berita       : {contoh_judul}")
print(f"Waktu Eksekusi     : {waktu_m1 * 1000:.2f} ms\n")

print("--- 1. Ringkasan Hasil Prediksi Model 1 (TF-IDF) ---")
print(ringkasan_m1)
print("\n--- 2. Ringkasan Acuan Manusia (Gold Reference) ---")
print(contoh_gold)

print("\n--- 3. Analisis Skor Kepentingan Tiap Kalimat ---")
df_skor = pd.DataFrame(rincian_skor)
display(df_skor)

In [ ]:
# ==============================================================================
# BAGIAN 3.3: VISUALISASI SELEKSI KALIMAT & DISTRIBUSI SKOR TF-IDF
# Pengembang: Asyam Sakhi Maulana (240535600363)
# ==============================================================================

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#2ca02c' if row['is_selected'] else '#aec7e8' for _, row in df_skor.iterrows()]

bars = ax.barh(
    [f"Kalimat #{r['original_pos']+1}" for _, r in df_skor.iterrows()][::-1],
    df_skor['score'].values[::-1],
    color=colors[::-1],
    edgecolor='black',
    linewidth=0.8
)

ax.set_title("Peringkat Skor Salience TF-IDF Tiap Kalimat (Hijau = Kalimat Terpilih)", fontsize=12, fontweight='bold')
ax.set_xlabel("Normalized Salience Score (TF-IDF / √length)", fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, f"{width:.3f}",
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Model 2: Extractive Summarization — TextRank Algorithm
**Penanggung Jawab:** **AGUSTINUS MARIA AGUNG HERIYANTO** (NIM: `240535607403`)  
**Peran & Fokus:** Konstruksi Graf Kemiripan Kalimat (Sentence Similarity Graph), Pembobotan Edge Cosine Similarity, Perhitungan Sentralitas Kalimat (PageRank), Seleksi Top-K Kalimat Sentral, & Rekonstruksi Ringkasan.

---

### 🧠 Landasan Teori Algoritma TextRank

Berbeda dengan **TF-IDF** yang menilai kalimat secara mandiri berdasarkan frekuensi kata, **TextRank** memodelkan seluruh artikel sebagai **graf tak berarah berbobot** $G = (V, E)$:
* **Node ($V$):** Setiap node mewakili satu kalimat $s_i$.
* **Edge ($E$):** Hubungan antar dua kalimat dengan bobot kemiripan semantik $w_{ij} = \text{CosineSimilarity}(s_i, s_j)$.
* **PageRank Scoring:** Kalimat yang memiliki hubungan kemiripan kuat dengan banyak kalimat penting lainnya akan menerima aliran skor (*vote*) yang tinggi:
  $$WS(v_i) = (1 - d) + d \times \sum_{v_j \in In(v_i)} \frac{w_{ji}}{\sum_{v_k \in Out(v_j)} w_{jk}} WS(v_j)$$
  *(dengan damping factor $d = 0.85$)*.

In [ ]:
# ==============================================================================
# BAGIAN 4.1: TEMPLATE MODEL 2 - TEXTRANK SUMMARIZER (JATAH AGUNG)
# Penanggung Jawab: Agustinus Maria Agung Heriyanto (240535607403)
# ==============================================================================

class TextRankSummarizer:
    """
    Model Extractive Summarization berbasis TextRank (Graph Cosine Similarity + PageRank).
    Penanggung Jawab: Agustinus Maria Agung Heriyanto (240535607403)
    
    📋 INSTRUKSI PENGERJAAN UNTUK AGUNG:
    Lengkapi blok kode bertanda # TODO (Agung) di bawah ini:
    1. Preprocessing segmentasi kalimat untuk graf.
    2. Vektorisasi TF-IDF & perhitungan Cosine Similarity matrix antarkalimat.
    3. Konstruksi graf berbobot NetworkX dan perhitungan skor sentralitas PageRank.
    4. Seleksi Top-K kalimat penting dan penyusunan urutan kronologis.
    """
    def __init__(self, stopwords=STOPWORDS_ID, damping=0.85, max_iter=200):
        self.stopwords = list(stopwords)
        self.damping = damping
        self.max_iter = max_iter
        
    def split_into_sentences(self, text):
        """
        [Modul Preprocessing Agung]
        Memecah artikel menjadi daftar kalimat untuk node graf.
        """
        raw_sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s.strip() for s in raw_sentences if len(s.strip().split()) >= 3]
        
    def summarize(self, text, num_sentences=2):
        """
        Ekstraksi ringkasan berbasis graf TextRank & PageRank.
        """
        # 1. Preprocessing: Segmentasi kalimat internal
        sentences = self.split_into_sentences(text)
        if len(sentences) <= num_sentences:
            return text.strip(), [{"original_pos": i, "sentence": s, "score": 1.0, "is_selected": True} for i, s in enumerate(sentences)]
            
        # ======================================================================
        # TODO (Agung): MULAI IMPLEMENTASI ALGORITMA TEXTRANK DI SINI
        # ======================================================================
        # Langkah 1 (Agung): Vektorisasi TF-IDF kalimat
        # vectorizer = TfidfVectorizer(stop_words=self.stopwords)
        # tfidf_matrix = vectorizer.fit_transform(sentences)
        
        # Langkah 2 (Agung): Hitung Adjacency Matrix kemiripan kosinus antar seluruh pasangan kalimat
        # sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
        # np.fill_diagonal(sim_matrix, 0)
        
        # Langkah 3 (Agung): Bangun graf berbobot menggunakan NetworkX
        # nx_graph = nx.from_numpy_array(sim_matrix)
        
        # Langkah 4 (Agung): Hitung sentralitas PageRank
        # scores = nx.pagerank(nx_graph, alpha=self.damping, max_iter=self.max_iter)
        
        # Langkah 5 (Agung): Urutkan kalimat berdasarkan PageRank & pilih top_indices
        # ranked_sentences = sorted(((scores[i], i) for i in range(len(sentences))), reverse=True)
        # top_indices = sorted([idx for _, idx in ranked_sentences[:num_sentences]])
        
        # ======================================================================
        # PLACEHOLDER SEMENTARA (Sebelum Agung melengkapi kodenya):
        # Mengembalikan kalimat awal sebagai placeholder agar notebook tetap bisa di-run.
        # ======================================================================
        top_indices = list(range(min(num_sentences, len(sentences))))
        selected_sentences = [sentences[i] for i in top_indices]
        summary_text = ' '.join(selected_sentences)
        
        ranking_details = [
            {
                "original_pos": i,
                "sentence": sentences[i],
                "score": 0.0,
                "is_selected": (i in top_indices)
            }
            for i in range(len(sentences))
        ]
        
        return summary_text, ranking_details

# Inisialisasi template Model 2 Agung
textrank_model = TextRankSummarizer()
print("📋 Template Model 2 (Agung) telah siap!")
print("ℹ️ Catatan: Model 2 masih berupa starter template. Menunggu implementasi TextRank dari Agung.")

# Uji coba inferensi starter template
ringkasan_m2, _ = textrank_model.summarize(contoh_artikel, num_sentences=2)
print("\n--- Output Percobaan Model 2 (Agung - Status: Starter Template) ---")
print(ringkasan_m2)


---
## 5. Evaluasi Komparatif & Metrik ROUGE (Slide 20 PDF)
**Kerangka Pengujian Komparatif Tim**  

Metrik **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)** adalah standar internasional untuk mengevaluasi kualitas ringkasan dengan membandingkan tumpang tindih n-gram antara ringkasan sistem (*candidate*) dan ringkasan acuan manusia (*gold reference*):
1. **ROUGE-1:** Mengukur overlap unigram (kata tunggal).
2. **ROUGE-2:** Mengukur overlap bigram (dua kata berurutan) untuk menilai kelancaran frasa.
3. **ROUGE-L:** Mengukur Longest Common Subsequence (LCS) untuk menilai struktur urutan kalimat.


In [ ]:
# ==============================================================================
# BAGIAN 5.1: FUNGSI PENGHITUNG METRIK ROUGE-1, ROUGE-2, & ROUGE-L
# ==============================================================================

def compute_rouge(candidate, reference):
    """
    Menghitung skor ROUGE-1, ROUGE-2, dan ROUGE-L secara presisi (Precision, Recall, F1).
    """
    def get_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        
    def lcs(seq1, seq2):
        m, n = len(seq1), len(seq2)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(1, m+1):
            for j in range(1, n+1):
                if seq1[i-1] == seq2[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        return dp[m][n]
        
    tok_cand = re.findall(r'\w+', candidate.lower())
    tok_ref  = re.findall(r'\w+', reference.lower())
    
    # ROUGE-1
    c1 = Counter(tok_cand)
    r1 = Counter(tok_ref)
    overlap_1 = sum((c1 & r1).values())
    rec_1 = overlap_1 / max(len(tok_ref), 1)
    prec_1 = overlap_1 / max(len(tok_cand), 1)
    f1_1 = 2 * (prec_1 * rec_1) / max((prec_1 + rec_1), 1e-8)
    
    # ROUGE-2
    c2 = Counter(get_ngrams(tok_cand, 2))
    r2 = Counter(get_ngrams(tok_ref, 2))
    overlap_2 = sum((c2 & r2).values())
    rec_2 = overlap_2 / max(len(tok_ref) - 1, 1)
    prec_2 = overlap_2 / max(len(tok_cand) - 1, 1)
    f1_2 = 2 * (prec_2 * rec_2) / max((prec_2 + rec_2), 1e-8)
    
    # ROUGE-L
    lcs_val = lcs(tok_cand, tok_ref)
    rec_l = lcs_val / max(len(tok_ref), 1)
    prec_l = lcs_val / max(len(tok_cand), 1)
    f1_l = 2 * (prec_l * rec_l) / max((prec_l + rec_l), 1e-8)
    
    return {
        "rouge1_f1": round(f1_1, 4),
        "rouge2_f1": round(f1_2, 4),
        "rougeL_f1": round(f1_l, 4)
    }

print("Fungsi evaluasi ROUGE terverifikasi!")

In [ ]:
# ==============================================================================
# BAGIAN 5.2: EVALUASI BENCHMARK PERBANDINGAN MODEL 1 VS MODEL 2 DI INDOSUM
# ==============================================================================

hasil_evaluasi = []

for idx, row in df_indosum.iterrows():
    art = row["article"]
    gold = row["gold_summary"]
    
    # Model 1: TF-IDF
    t0 = time.time()
    pred_m1, _ = tfidf_model.summarize(art, num_sentences=2)
    waktu_m1 = time.time() - t0
    r_m1 = compute_rouge(pred_m1, gold)
    
    # Model 2: TextRank
    t0 = time.time()
    pred_m2, _ = textrank_model.summarize(art, num_sentences=2)
    waktu_m2 = time.time() - t0
    r_m2 = compute_rouge(pred_m2, gold)
    
    hasil_evaluasi.append({
        "id": row["id"],
        "category": row["category"],
        "m1_rouge1": r_m1["rouge1_f1"],
        "m1_rouge2": r_m1["rouge2_f1"],
        "m1_rougeL": r_m1["rougeL_f1"],
        "m1_time_ms": waktu_m1 * 1000,
        "m2_rouge1": r_m2["rouge1_f1"],
        "m2_rouge2": r_m2["rouge2_f1"],
        "m2_rougeL": r_m2["rougeL_f1"],
        "m2_time_ms": waktu_m2 * 1000,
        "pred_m1": pred_m1,
        "pred_m2": pred_m2,
        "gold": gold
    })

df_hasil = pd.DataFrame(hasil_evaluasi)

# Tabel Rangkuman Rata-Rata Performa
tabel_ringkasan = pd.DataFrame({
    "Metrik Evaluasi": ["ROUGE-1 (F1)", "ROUGE-2 (F1)", "ROUGE-L (F1)", "Waktu Komputasi Rata-rata (ms)"],
    "Model 1: TF-IDF Sentence Ranking (Asyam)": [
        round(df_hasil["m1_rouge1"].mean(), 4),
        round(df_hasil["m1_rouge2"].mean(), 4),
        round(df_hasil["m1_rougeL"].mean(), 4),
        round(df_hasil["m1_time_ms"].mean(), 2)
    ],
    "Model 2: TextRank Algorithm (Agung)": [
        round(df_hasil["m2_rouge1"].mean(), 4),
        round(df_hasil["m2_rouge2"].mean(), 4),
        round(df_hasil["m2_rougeL"].mean(), 4),
        round(df_hasil["m2_time_ms"].mean(), 2)
    ]
})

print("=== TABEL PERBANDINGAN KOMPARATIF TEXT SUMMARIZATION (SLIDE 20 PDF) ===")
display(tabel_ringkasan)

In [ ]:
# ==============================================================================
# BAGIAN 5.3: VISUALISASI METRIK ROUGE & WAKTU KOMPUTASI
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Subplot 1: Skor ROUGE
metriks = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
m1_scores = [df_hasil["m1_rouge1"].mean(), df_hasil["m1_rouge2"].mean(), df_hasil["m1_rougeL"].mean()]
m2_scores = [df_hasil["m2_rouge1"].mean(), df_hasil["m2_rouge2"].mean(), df_hasil["m2_rougeL"].mean()]

x = np.arange(len(metriks))
axes[0].bar(x - 0.2, m1_scores, 0.4, label="Model 1 (TF-IDF)", color="#1f77b4")
axes[0].bar(x + 0.2, m2_scores, 0.4, label="Model 2 (TextRank)", color="#ff7f0e")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metriks, fontsize=11, fontweight="bold")
axes[0].set_ylabel("F1-Score", fontsize=10)
axes[0].set_title("Perbandingan Skor ROUGE pada Korpus IndoSum", fontsize=12, fontweight="bold")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# Subplot 2: Waktu Komputasi
axes[1].bar(["Model 1: TF-IDF", "Model 2: TextRank"], [df_hasil["m1_time_ms"].mean(), df_hasil["m2_time_ms"].mean()],
            color=["#1f77b4", "#ff7f0e"], width=0.5, edgecolor="black")
axes[1].set_ylabel("Latensi Proses (ms)", fontsize=10)
axes[1].set_title("Waktu Pemrosesan Rata-Rata per Artikel", fontsize=12, fontweight="bold")
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

---
## 6. Analisis Kualitas Ringkasan & Kesimpulan Akhir
**Penanggung Jawab:** **CHRISTIAN LIONEL DAUN DARMA** (NIM: `240535604624`) — *Tugas: Penulisan Kesimpulan*  

### 💬 Pembahasan Pertanyaan Kritis Slide 20 PDF:
> **"Apakah model dengan skor ROUGE tertinggi selalu menghasilkan ringkasan yang paling baik secara makna?"**

**Jawaban & Analisis Kesimpulan:**  
**TIDAK SELALU.**  
1. **Keterbatasan Metrik ROUGE:** ROUGE bekerja secara leksikal (*string matching* / overlap n-gram). Jika ringkasan sistem menggunakan sinonim yang berbeda dari ringkasan manusia (misal: *"menurunkan"* vs *"memangkas"*, atau *"pemerintah"* vs *"presiden"*), skor ROUGE bisa rendah padahal maknanya sama persis.
2. **Koherensi dan Aliran Makna:** Kalimat dengan kata kunci relevan tinggi yang dipilih oleh model ekstraktif belum tentu menyatu secara logis jika dibaca berurutan, terutama jika memotong konteks kalimat sebelumnya (*pronoun reference problem*).
3. **Kelebihan TF-IDF (Asyam):** Sangat cepat (komputasi matriks langsung ~1 ms) dan andal menangkap kata kunci spesifik dokumen. Kelemahannya tidak memperhitungkan relasi semantik antar kalimat.
4. **Kelebihan TextRank (Agung):** Memodelkan artikel sebagai graf jejaring interaksi kalimat sehingga mampu memilih kalimat sentral dokumen. Kelemahannya komputasi lebih lambat ($O(N^2)$) pada dokumen berita panjang.


In [ ]:
# ==============================================================================
# BAGIAN 6.1: CONTOH RINGKASAN TERBAIK DAN TERBURUK (QUALITATIVE ERROR ANALYSIS)
# ==============================================================================

# Ringkasan dengan ROUGE-1 Tertinggi vs Terendah pada Model 1
best_idx = df_hasil["m1_rouge1"].idxmax()
worst_idx = df_hasil["m1_rouge1"].idxmin()

print("="*80)
print(f"🌟 CONTOH RINGKASAN TERBAIK (Artikel #{best_idx+1} - ROUGE-1: {df_hasil.loc[best_idx, 'm1_rouge1']:.4f})")
print("="*80)
print("Judul       :", df_indosum.loc[best_idx, "title"])
print("Ringkasan M1:", df_hasil.loc[best_idx, "pred_m1"])
print("Gold Summary:", df_hasil.loc[best_idx, "gold"])

print()
print("="*80)
print(f"⚠️ CONTOH RINGKASAN TERBURUK (Artikel #{worst_idx+1} - ROUGE-1: {df_hasil.loc[worst_idx, 'm1_rouge1']:.4f})")
print("="*80)
print("Judul       :", df_indosum.loc[worst_idx, "title"])
print("Ringkasan M1:", df_hasil.loc[worst_idx, "pred_m1"])
print("Gold Summary:", df_hasil.loc[worst_idx, "gold"])


### ⚖️ Perbandingan Kelemahan & Kelebihan Masing-Masing Metode

| Aspek Perbandingan | Model 1: TF-IDF Sentence Ranking (Asyam) | Model 2: TextRank Algorithm (Agung) |
| :--- | :--- | :--- |
| **Prinsip Dasar** | Bobot frekuensi dan kelangkaan leksikal kata dalam kalimat. | Graf kemiripan antar-kalimat & sentralitas PageRank. |
| **Kecepatan Proses** | **Sangat Cepat** (komputasi matriks langsung, latensi ~1–3 ms). | **Sedikit Lebih Lambat** (perlu iterasi konvergensi PageRank dan graf pasangan kalimat $O(N^2)$). |
| **Kekuatan Utama** | Sangat andal menangkap kalimat yang memuat kata kunci spesifik dan dominan dalam dokumen. | Unggul dalam memilih kalimat yang mewakili inti konsensus keseluruhan dokumen. |
| **Kelemahan Utama** | Cenderung bias terhadap kalimat yang memiliki banyak kata repetitif jika tidak dinormalisasi. | Dapat kesulitan jika antar-kalimat memiliki kemiripan kosakata yang sangat rendah (*sparse graph*). |
| **Kebutuhan Training** | Tanpa training (*unsupervised heuristic*). | Tanpa training (*graph-based unsupervised*). |

---
## 7. Bonus: Perbandingan Extractive vs. Abstractive Summarization
Membandingkan hasil peringkasan ekstraktif (TF-IDF & TextRank) dengan peringkasan **Abstraktif** menggunakan pre-trained Transformer Bahasa Indonesia.

In [ ]:
# ==============================================================================
# BAGIAN 7: BONUS PRE-TRAINED ABSTRACTIVE SUMMARIZATION (T5 / IndoBART)
# ==============================================================================

# Komparasi kualitatif ekstraktif vs abstraktif
print("=== KOMPARASI KONSEPTUAL: EKSTRAKTIF VS ABSTRAKTIF ===")
print("• Ekstraktif (Model 1 & 2): Memilih kalimat penting dari teks asli tanpa mengubah kata.")
print("  Kelebihan : 100% akurat secara fakta, tidak berhalusinasi, tanpa GPU berat.")
print("  Kekurangan: Kalimat bisa kaku dan tidak ringkas seperti gaya tulisan manusia.\n")
print("• Abstraktif (Transformer T5/BART): Menulis ulang gagasan utama dengan kalimat baru.")
print("  Kelebihan : Alur bahasa sangat alami, parafrase luwes mirip tulisan jurnalis.")
print("  Kekurangan: Membutuhkan model besar, waktu inferensi lebih tinggi, berisiko halusinasi fakta.")